In [1]:
# Uninstall current version and install the new local version
import sys
import subprocess

# Uninstall current ocr-data-toolkit
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "ocr-data-toolkit", "-y"])

# Install the new version from the local folder
subprocess.check_call([sys.executable, "-m", "pip", "install", r"/home/user/Desktop/Nomocrat/ocr-data-toolkit"])

print("Successfully uninstalled old version and installed new version!")

Found existing installation: ocr-data-toolkit 0.1.2
Uninstalling ocr-data-toolkit-0.1.2:
  Successfully uninstalled ocr-data-toolkit-0.1.2
Processing ./.
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for ocr-data-toolkit: filename=ocr_data_toolkit-0.1.2-py3-none-any.whl size=28051354 sha256=8b0f03b876b83d783159cc8be953a8c48689056ab7762c55dbdf958a646d5161
  Stored in directory: /home/user/.cache/pip/wheels/e7/53/b8/d230ee403a428a7794abf83cdec4d330dbfe8e700665461c1e
Successfully built ocr-data-toolkit
Successfully uninstalled old version and installed new version!


References for Synthetic Data

- OCR synthetic data github: https://github.com/NaumanHSA/ocr-data-toolkit.git 
- Mlt word book: https://github.com/mtanti/maltese-word-lists/blob/main/dash_apostrophe-lowercase-km4.2/dash_apostrophe-lowercase-km4.2-v1.txt 

# Arranged with maltese fonts

In [2]:
import sys, os, glob
from pathlib import Path

sys.path.insert(0, r"/home/user/Desktop/Nomocrat/ocr-data-toolkit")

from ocr_data_toolkit import ODT
from ocr_data_toolkit.generators import DataGenerator, TextGenerator
import textwrap, random, hyphen

font_dir = r"/home/user/Desktop/Nomocrat/ocr-data-toolkit/ocr_data_toolkit/data/fonts/mlt"
font_paths = glob.glob(os.path.join(font_dir, "*.ttf"))

SPLIT_TEXT_PATH = Path(r"/home/user/Desktop/Nomocrat/text_samples/train_sample_32000_split.txt")
TEXT_PATH = SPLIT_TEXT_PATH
print(f"Using text source: {TEXT_PATH}")

class MyGenerator(TextGenerator):
    def __init__(self, seed):
        super().__init__("en")  # keep as en since installed package lacks mlt
        self.rng = random.Random(seed)
        with TEXT_PATH.open(encoding="utf-8") as f:
            self.texts = [line.strip() for line in f if line.strip()]
        self.rng.shuffle(self.texts)
        self.texts_iter = iter(self.texts)
        self.h_en = hyphen.Hyphenator("en_US")  # optional

    def __call__(self):
        try:
            gt_text = next(self.texts_iter)
        except StopIteration:
            self.rng.shuffle(self.texts)
            self.texts_iter = iter(self.texts)
            gt_text = next(self.texts_iter)
        text_width = self.rng.randint(30, 100)
        # im_text = textwrap.fill(gt_text, width=text_width)
        im_text = gt_text  # Keep as is for now; ODT will handle line-breaking if needed
        return gt_text, im_text

odt = ODT(
    data_generator=DataGenerator(MyGenerator(seed=0), 
                                 font_paths=font_paths, 
                                 background_paths=r"/home/user/Desktop/Nomocrat/ocr-data-toolkit/ocr_data_toolkit/data/backgrounds",  
                                #  background_paths=[],  # Empty list = plain white backgrounds only
                                 augmentation_config={
                                    "moire_prob": 0,
                                    "blur_probs": {
                                        "gaussian": 0.2,
                                        "custom_blurs": 0
                                    },
                                    "perspective_transform_prob": 0,
                                    "opacity_prob": 0.1,
                                    "brightness_range": (0.92, 1.08),
                                    "letter_spacing": 0,
                                    "margin_x": (0, 0),
                                    "margin_y": (0, 0)
                                }
                                ),
    train_test_ratio=0.8
)

INFO:ocr_data_toolkit.generators.data_generator:Fonts loaded.
INFO:ocr_data_toolkit.generators.data_generator:Backgrounds loaded.


Using text source: /home/user/Desktop/Nomocrat/text_samples/train_sample_32000_split.txt


In [3]:
# text, image = odt.generate_single_image()
# image.save('sample.png')
# print(text)

odt.generate_training_data(num_samples=200000)

INFO:ocr_data_toolkit.odt:Generating images...


04/19 07:03 :        0 /    50000 (  0.00%): Job 1 
04/19 07:03 :        0 /    50000 (  0.00%): Job 2 
04/19 07:03 :        0 /    50000 (  0.00%): Job 3 
04/19 07:03 :        0 /    50000 (  0.00%): Job 0 
04/19 07:04 :      570 /    50000 (  1.14%): Job 1 
04/19 07:04 :      562 /    50000 (  1.12%): Job 3 
04/19 07:04 :      577 /    50000 (  1.15%): Job 2 
04/19 07:04 :      603 /    50000 (  1.21%): Job 0 
04/19 07:05 :     1099 /    50000 (  2.20%): Job 3 
04/19 07:05 :     1132 /    50000 (  2.26%): Job 2 
04/19 07:05 :     1173 /    50000 (  2.35%): Job 0 
04/19 07:05 :     1140 /    50000 (  2.28%): Job 1 
04/19 07:06 :     1664 /    50000 (  3.33%): Job 3 
04/19 07:06 :     1701 /    50000 (  3.40%): Job 2 
04/19 07:06 :     1746 /    50000 (  3.49%): Job 0 
04/19 07:06 :     1697 /    50000 (  3.39%): Job 1 
04/19 07:07 :     2237 /    50000 (  4.47%): Job 3 
04/19 07:07 :     2256 /    50000 (  4.51%): Job 2 
04/19 07:07 :     2340 /    50000 (  4.68%): Job 0 
04/19 07:07 

INFO:ocr_data_toolkit.odt:Finalizing...


04/19 08:31 :    50000 /    50000 (100.00%): Job 2 


INFO:ocr_data_toolkit.odt:Number of Train records 0
INFO:ocr_data_toolkit.odt:Number of Test records 200000
